# Advanced Machine Learning with Scikit-Learn

**Module 02: Machine Learning**  
**Objective**: Master production-ready ML with scikit-learn

Now that you understand ML algorithms from first principles, let's use scikit-learn for:
- Production-quality implementations
- Advanced ensembles (Random Forests, Gradient Boosting)
- Model selection and hyperparameter tuning
- Pipelines and preprocessing
- Model evaluation and validation

## What You'll Learn
1. Scikit-learn API patterns
2. Random Forests
3. Gradient Boosting (XGBoost, LightGBM)
4. Support Vector Machines
5. Cross-validation and hyperparameter tuning
6. Feature engineering pipelines
7. Model evaluation metrics
8. Handling imbalanced data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification, load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_style('whitegrid')

## 1. Scikit-Learn API Patterns

All estimators follow the same interface:
- `fit(X, y)`: Train the model
- `predict(X)`: Make predictions
- `score(X, y)`: Evaluate the model
- `get_params()`: Get hyperparameters
- `set_params(**params)`: Set hyperparameters

In [ ]:
# Load dataset
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15,
                           n_redundant=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Example: Logistic Regression
model = LogisticRegression(max_iter=1000, random_state=42)

# Fit
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

# Score
train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)

print(f"Scikit-Learn API Pattern:")
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"\nPredictions shape: {y_pred.shape}")
print(f"Probabilities shape: {y_proba.shape}")
print(f"\nModel parameters: {list(model.get_params().keys())[:5]}...")

## 2. Random Forests

**Ensemble method**: Combines multiple decision trees trained on random subsets of data and features.

**Advantages**:
- Reduces overfitting compared to single decision tree
- Handles nonlinear relationships
- Feature importance for free
- Robust to outliers

In [ ]:
# Load breast cancer dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Classes: {np.unique(y)} (0=malignant, 1=benign)\n")

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5,
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Predictions
y_pred_train = rf.predict(X_train)
y_pred_test = rf.predict(X_test)
y_proba_test = rf.predict_proba(X_test)[:, 1]

# Evaluation
train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)
test_auc = roc_auc_score(y_test, y_proba_test)

print(f"Random Forest Results:")
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test AUC-ROC: {test_auc:.4f}\n")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Most Important Features:")
print(feature_importance.head(10))

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance.head(15)['feature'], feature_importance.head(15)['importance'])
plt.xlabel('Importance')
plt.title('Random Forest: Top 15 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Random Forest (AUC = {test_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Gradient Boosting

**Idea**: Build trees sequentially, each correcting errors of previous trees.

**Popular implementations**:
- `GradientBoostingClassifier` (sklearn)
- `XGBoost` (powerful, widely used)
- `LightGBM` (fast, handles large datasets)
- `CatBoost` (handles categorical features)

In [ ]:
# Gradient Boosting (sklearn)
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3,
                                random_state=42)
gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)
y_proba_gb = gb.predict_proba(X_test)[:, 1]

gb_acc = accuracy_score(y_test, y_pred_gb)
gb_auc = roc_auc_score(y_test, y_proba_gb)

print(f"Gradient Boosting Results:")
print(f"Test Accuracy: {gb_acc:.4f}")
print(f"Test AUC-ROC: {gb_auc:.4f}\n")

# Compare multiple models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'Naive Bayes': GaussianNB()
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba) if y_proba is not None else None
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'AUC-ROC': auc
    })

results_df = pd.DataFrame(results).sort_values('AUC-ROC', ascending=False)
print("Model Comparison:")
print(results_df.to_string(index=False))

# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
results_df_sorted = results_df.sort_values('Accuracy')
ax1.barh(results_df_sorted['Model'], results_df_sorted['Accuracy'])
ax1.set_xlabel('Accuracy')
ax1.set_title('Model Comparison: Accuracy')
ax1.set_xlim(0.9, 1.0)

# AUC comparison
results_df_sorted = results_df.sort_values('AUC-ROC')
ax2.barh(results_df_sorted['Model'], results_df_sorted['AUC-ROC'])
ax2.set_xlabel('AUC-ROC')
ax2.set_title('Model Comparison: AUC-ROC')
ax2.set_xlim(0.9, 1.0)

plt.tight_layout()
plt.show()

## 4. Cross-Validation

**Problem**: Single train/test split is noisy.

**Solution**: K-Fold Cross-Validation
- Split data into K folds
- Train on K-1 folds, test on 1
- Repeat K times
- Average results

In [ ]:
from sklearn.model_selection import cross_validate

# Cross-validation with multiple metrics
model = RandomForestClassifier(n_estimators=100, random_state=42)

cv_results = cross_validate(
    model, X, y, cv=5,
    scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
    return_train_score=True,
    n_jobs=-1
)

print("5-Fold Cross-Validation Results:")
print(f"Train Accuracy: {cv_results['train_accuracy'].mean():.4f} ± {cv_results['train_accuracy'].std():.4f}")
print(f"Test Accuracy:  {cv_results['test_accuracy'].mean():.4f} ± {cv_results['test_accuracy'].std():.4f}")
print(f"Test Precision: {cv_results['test_precision'].mean():.4f} ± {cv_results['test_precision'].std():.4f}")
print(f"Test Recall:    {cv_results['test_recall'].mean():.4f} ± {cv_results['test_recall'].std():.4f}")
print(f"Test F1:        {cv_results['test_f1'].mean():.4f} ± {cv_results['test_f1'].std():.4f}")
print(f"Test AUC-ROC:   {cv_results['test_roc_auc'].mean():.4f} ± {cv_results['test_roc_auc'].std():.4f}")

# Learning curve
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    model, X, y, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy'
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, label='Training score', marker='o')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.3)
plt.plot(train_sizes, val_mean, label='Validation score', marker='o')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.3)
plt.xlabel('Training Set Size')
plt.ylabel('Accuracy')
plt.title('Learning Curve: Random Forest')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Hyperparameter Tuning

**Grid Search**: Try all combinations

**Random Search**: Sample random combinations (faster, often better)

In [ ]:
# Grid Search
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    rf, param_grid, cv=5, scoring='roc_auc',
    n_jobs=-1, verbose=0
)

print("Running Grid Search (this may take a minute)...")
grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

# Test set performance
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Test AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}")

# Random Search (faster for large parameter spaces)
from scipy.stats import randint, uniform

param_distributions = {
    'n_estimators': randint(50, 300),
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', None]
}

random_search = RandomizedSearchCV(
    rf, param_distributions, n_iter=50, cv=5,
    scoring='roc_auc', n_jobs=-1, random_state=42, verbose=0
)

print("\nRunning Random Search...")
random_search.fit(X_train, y_train)

print(f"Best parameters: {random_search.best_params_}")
print(f"Best CV score: {random_search.best_score_:.4f}")

# Plot hyperparameter importance
cv_results = pd.DataFrame(random_search.cv_results_)
param_cols = [col for col in cv_results.columns if col.startswith('param_')]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for idx, param in enumerate(param_cols[:6]):
    param_name = param.replace('param_', '')
    ax = axes[idx]
    
    # Handle numeric vs categorical
    if cv_results[param].dtype in [int, float]:
        ax.scatter(cv_results[param], cv_results['mean_test_score'], alpha=0.5)
        ax.set_xlabel(param_name)
    else:
        cv_results.boxplot(column='mean_test_score', by=param, ax=ax)
        ax.set_xlabel(param_name)
    
    ax.set_ylabel('CV Score' if idx % 3 == 0 else '')
    ax.set_title(f'{param_name}')

plt.tight_layout()
plt.show()

## 6. Pipelines (Production-Ready ML)

**Pipelines**: Chain preprocessing and modeling steps.

**Benefits**:
- Prevents data leakage
- Cleaner code
- Easy to deploy

In [ ]:
# Create pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Fit pipeline (scaling applied automatically)
pipeline.fit(X_train, y_train)

# Predict (scaling applied automatically)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(f"Pipeline Results:")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Test AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}\n")

# Grid search with pipeline
param_grid = {
    'scaler': [StandardScaler(), MinMaxScaler(), RobustScaler(), None],
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [5, 10, None]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best pipeline configuration:")
print(f"Scaler: {grid_search.best_params_['scaler']}")
print(f"n_estimators: {grid_search.best_params_['classifier__n_estimators']}")
print(f"max_depth: {grid_search.best_params_['classifier__max_depth']}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

## 7. Handling Imbalanced Data

**Problem**: When one class dominates (e.g., fraud detection: 99% normal, 1% fraud)

**Solutions**:
1. Class weights
2. Resampling (oversample minority, undersample majority)
3. SMOTE (Synthetic Minority Over-sampling Technique)
4. Different metrics (use F1, precision, recall, not accuracy)

In [ ]:
# Create imbalanced dataset
X_imb, y_imb = make_classification(n_samples=1000, n_features=20, n_informative=15,
                                   weights=[0.95, 0.05], random_state=42)

X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb
)

print(f"Imbalanced dataset:")
print(f"Class distribution: {np.bincount(y_train_imb)}")
print(f"Class 0: {np.bincount(y_train_imb)[0]/len(y_train_imb):.1%}")
print(f"Class 1: {np.bincount(y_train_imb)[1]/len(y_train_imb):.1%}\n")

# Naive approach (disaster!)
rf_naive = RandomForestClassifier(n_estimators=100, random_state=42)
rf_naive.fit(X_train_imb, y_train_imb)
y_pred_naive = rf_naive.predict(X_test_imb)

print("Naive approach:")
print(f"Accuracy: {accuracy_score(y_test_imb, y_pred_naive):.4f} (misleading!)")
print(f"Precision: {precision_score(y_test_imb, y_pred_naive):.4f}")
print(f"Recall: {recall_score(y_test_imb, y_pred_naive):.4f}")
print(f"F1: {f1_score(y_test_imb, y_pred_naive):.4f}\n")

# Solution 1: Class weights
rf_weighted = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_weighted.fit(X_train_imb, y_train_imb)
y_pred_weighted = rf_weighted.predict(X_test_imb)

print("With class weights:")
print(f"Accuracy: {accuracy_score(y_test_imb, y_pred_weighted):.4f}")
print(f"Precision: {precision_score(y_test_imb, y_pred_weighted):.4f}")
print(f"Recall: {recall_score(y_test_imb, y_pred_weighted):.4f}")
print(f"F1: {f1_score(y_test_imb, y_pred_weighted):.4f}\n")

# Confusion matrices
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

cm_naive = confusion_matrix(y_test_imb, y_pred_naive)
sns.heatmap(cm_naive, annot=True, fmt='d', cmap='Blues', ax=ax1)
ax1.set_title('Confusion Matrix: Naive')
ax1.set_ylabel('True')
ax1.set_xlabel('Predicted')

cm_weighted = confusion_matrix(y_test_imb, y_pred_weighted)
sns.heatmap(cm_weighted, annot=True, fmt='d', cmap='Blues', ax=ax2)
ax2.set_title('Confusion Matrix: Class Weights')
ax2.set_ylabel('True')
ax2.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

# Classification report
print("Classification Report (Class Weights):")
print(classification_report(y_test_imb, y_pred_weighted))

## 8. Feature Engineering

**Critical for model performance!**

Common techniques:
- Polynomial features
- Interaction terms
- Binning
- Log transforms
- Domain-specific features

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Load diabetes dataset (regression)
diabetes = load_diabetes()
X_diab = diabetes.data
y_diab = diabetes.target

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_diab, y_diab, test_size=0.2, random_state=42
)

print(f"Diabetes dataset: {X_diab.shape[0]} samples, {X_diab.shape[1]} features\n")

# Baseline: Linear regression
from sklearn.linear_model import Ridge

baseline = Ridge(alpha=1.0)
baseline.fit(X_train_d, y_train_d)
y_pred_baseline = baseline.predict(X_test_d)

baseline_mse = mean_squared_error(y_test_d, y_pred_baseline)
baseline_r2 = r2_score(y_test_d, y_pred_baseline)

print(f"Baseline (Ridge):")
print(f"Test MSE: {baseline_mse:.2f}")
print(f"Test R²: {baseline_r2:.4f}\n")

# Feature engineering: Polynomial features
poly_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])

poly_pipeline.fit(X_train_d, y_train_d)
y_pred_poly = poly_pipeline.predict(X_test_d)

poly_mse = mean_squared_error(y_test_d, y_pred_poly)
poly_r2 = r2_score(y_test_d, y_pred_poly)

print(f"With Polynomial Features:")
print(f"Features expanded: {X_train_d.shape[1]} → {poly_pipeline.named_steps['poly'].transform(X_train_d).shape[1]}")
print(f"Test MSE: {poly_mse:.2f}")
print(f"Test R²: {poly_r2:.4f}")
print(f"Improvement: {(baseline_mse - poly_mse)/baseline_mse:.1%} reduction in MSE")

# Plot predictions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.scatter(y_test_d, y_pred_baseline, alpha=0.6)
ax1.plot([y_test_d.min(), y_test_d.max()], [y_test_d.min(), y_test_d.max()], 'r--')
ax1.set_xlabel('True Values')
ax1.set_ylabel('Predictions')
ax1.set_title(f'Baseline (R²={baseline_r2:.3f})')
ax1.grid(True, alpha=0.3)

ax2.scatter(y_test_d, y_pred_poly, alpha=0.6)
ax2.plot([y_test_d.min(), y_test_d.max()], [y_test_d.min(), y_test_d.max()], 'r--')
ax2.set_xlabel('True Values')
ax2.set_ylabel('Predictions')
ax2.set_title(f'Polynomial Features (R²={poly_r2:.3f})')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Model Persistence (Save/Load)

In [ ]:
import joblib

# Save model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

joblib.dump(model, 'random_forest_model.pkl')
print("Model saved to 'random_forest_model.pkl'\n")

# Load model
loaded_model = joblib.load('random_forest_model. pkl')
y_pred_loaded = loaded_model.predict(X_test)

print(f"Loaded model accuracy: {accuracy_score(y_test, y_pred_loaded):.4f}")
print("Model loaded and working correctly!")

# Save pipeline
pipeline_to_save = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
pipeline_to_save.fit(X_train, y_train)

joblib.dump(pipeline_to_save, 'pipeline.pkl')
print("\nPipeline saved to 'pipeline.pkl'")

## 10. Best Practices Summary

### Model Selection
1. **Start simple**: Logistic/Linear Regression baseline
2. **Try ensembles**: Random Forest, Gradient Boosting
3. **Deep learning**: Only if you have lots of data

### Validation
1. **Always use cross-validation** for model selection
2. **Hold out test set** for final evaluation
3. **Stratified splits** for classification

### Hyperparameter Tuning
1. **Random Search first** (faster exploration)
2. **Grid Search** for fine-tuning
3. **Use pipelines** to prevent data leakage

### Feature Engineering
1. **Domain knowledge** is key
2. **Feature scaling** helps most models
3. **Polynomial features** can improve linear models

### Imbalanced Data
1. **Don't use accuracy** as metric
2. **Class weights** or **resampling**
3. **Focus on precision/recall trade-off**

### Production
1. **Save pipelines**, not just models
2. **Version control** models (MLflow, DVC)
3. **Monitor performance** in production

## Exercises

### Exercise 1: Build Complete ML Pipeline
Create a pipeline that:
1. Handles missing values
2. Scales features
3. Selects top k features
4. Trains classifier
5. Tunes hyperparameters

In [ ]:
# TODO: Implement complete pipeline
# from sklearn.impute import SimpleImputer
# from sklearn.feature_selection import SelectKBest, f_classif
# 
# pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy='median')),
#     ('scaler', ...),
#     ('feature_selection', ...),
#     ('classifier', ...)
# ])

### Exercise 2: Implement Stacking Ensemble
Combine predictions from multiple models using a meta-learner.

In [ ]:
# TODO: Implement stacking
# from sklearn.ensemble import StackingClassifier
# 
# base_learners = [
#     ('rf', RandomForestClassifier()),
#     ('gb', GradientBoostingClassifier()),
#     ('svm', SVC(probability=True))
# ]
# 
# stacking_clf = StackingClassifier(
#     estimators=base_learners,
#     final_estimator=LogisticRegression()
# )

### Exercise 3: Implement Custom Transformer
Build a custom transformer that adds domain-specific features.

In [ ]:
# TODO: Implement custom transformer
# from sklearn.base import BaseEstimator, TransformerMixin
# 
# class CustomFeatureAdder(BaseEstimator, TransformerMixin):
#     def fit(self, X, y=None):
#         return self
#     
#     def transform(self, X):
#         # Add your custom features here
#         pass

## Summary

You've mastered:
- ✅ Scikit-learn API and patterns
- ✅ Random Forests and Gradient Boosting
- ✅ Cross-validation and hyperparameter tuning
- ✅ Production-ready pipelines
- ✅ Handling imbalanced data
- ✅ Feature engineering techniques
- ✅ Model evaluation metrics
- ✅ Model persistence

**Next Module**: Deep Learning from scratch with neural networks!